# NOTEBOOK 3.5(a) Spark Broadcast Variables and Accumulators

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

26/01/19 16:49:24 WARN Utils: Your hostname, PC25. resolves to a loopback address: 127.0.1.1; using 192.168.76.195 instead (on interface eth0)
26/01/19 16:49:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/19 16:49:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Broadcast Variables
Broadcast variables in PySpark allow you to efficiently share large read-only data across all worker nodes rather than sending it with every task. This reduces data transfer overhead and improves performance.

- **sc.broadcast()** is used to create a broadcast variable.
- The **.value property** gives access to the broadcasted data.

In [2]:
# Declare a dictionary to map the country code to the country name. 
country_map = {
    "US": "United States",
    "UK": "United Kingdom",
    "IN": "India"
}

### Create a broadcast variable 
The broadcast variable **broadcast_country_map** for the country code dictionary. This approach avoids sending the dictionary to every executor task individually, saving memory and network bandwidth.

In [3]:
broadcast_country_map = sc.broadcast(country_map)

In [4]:
# The enrich_with_country() function adds the country name to the user tuple
def enrich_with_country(user):
    user_id, name, country_code = user
    country_name = broadcast_country_map.value.get(country_code, "Unknown")
    return (user_id, name, country_code, country_name)

In [5]:
# Sample data
users = [
    (1, "Alice", "US"),
    (2, "Bob", "UK"),
    (3, "Charlie", "IN"),
    (4, "David", "US")
]

user_rdd = sc.parallelize(users)

In [6]:
enriched_rdd = user_rdd.map(enrich_with_country)

In [7]:
for record in enriched_rdd.collect():
    print(record)

(1, 'Alice', 'US', 'United States')
(2, 'Bob', 'UK', 'United Kingdom')
(3, 'Charlie', 'IN', 'India')
(4, 'David', 'US', 'United States')


## 2. Accumulators
Accumulators in PySpark are write-only shared variables used to perform aggregations across executors (e.g., counters or sums). They're commonly used for debugging or monitoring, such as counting corrupt records, missing fields, etc.

In [8]:
# Sample data
users = [
    (1, "Alice"),
    (2, ""),
    (3, "Charlie"),
    (4, ""),
    (5, "Eve")
]

user_rdd = sc.parallelize(users)

In [9]:
# Define an accumulator for counting blank names
blank_name_count = sc.accumulator(0)

In [10]:
# The check_name() function checks for blank names and updates the accumulator
def check_name(record):
    user_id, name = record
    if name.strip() == "":
        blank_name_count.add(1)
    return record

In [11]:
processed_rdd = user_rdd.map(check_name)
processed_rdd.collect()

[(1, 'Alice'), (2, ''), (3, 'Charlie'), (4, ''), (5, 'Eve')]

In [12]:
print(f"Number of users with blank names: {blank_name_count.value}")

Number of users with blank names: 2


In [13]:
spark.stop()